In [100]:
import pandas as pd
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
print("Working ✅")

Working ✅


In [101]:
df=pd.read_csv("../src/data/raw/bank-additional-full.csv",sep=";")
print(df.columns)

Index(['age', 'job', 'marital', 'education', 'default', 'housing', 'loan',
       'contact', 'month', 'day_of_week', 'duration', 'campaign', 'pdays',
       'previous', 'poutcome', 'emp.var.rate', 'cons.price.idx',
       'cons.conf.idx', 'euribor3m', 'nr.employed', 'y'],
      dtype='object')


In [102]:
import os
print(os.getcwd())

c:\Users\Shravani\Desktop\Projects\mlops-drift-monitor\notebooks


In [103]:
X=df.drop("y",axis=1)
#convert target variable to binary
y=df["y"].str.strip().map({"yes":1,"no":0})

In [104]:
print(X.shape)
print(y.shape)
print(y.head())
print(y.unique())

(41188, 20)
(41188,)
0    0
1    0
2    0
3    0
4    0
Name: y, dtype: int64
[0 1]


In [105]:
#Give me all columns that are numeric
num_cols=X.select_dtypes(include=["int64","float64"]).columns
print(num_cols)

#Give me all columns that are categorical
cat_cols=X.select_dtypes(include=["object"]).columns
print(cat_cols)

Index(['age', 'duration', 'campaign', 'pdays', 'previous', 'emp.var.rate',
       'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed'],
      dtype='object')
Index(['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact',
       'month', 'day_of_week', 'poutcome'],
      dtype='object')


In [106]:
model = GradientBoostingClassifier()
model_name = "GradientBoostingClassifier"

#We will now build a preprocessor
#Here ColumnTransformer allows us to apply different transformations to different columns
#handle_unknown="ignore" is used to ignore any unknown categories during transformation
#Pipeline allows us to chain multiple steps together, in this case preprocessing and modeling

preprocessor=ColumnTransformer([
    ("num",StandardScaler(),num_cols),
    ("cat",OneHotEncoder(handle_unknown="ignore"),cat_cols)
])

pipeline=Pipeline([
    ("preprocessing",preprocessor),
    ("model",model)
])

In [107]:
#Split the data into train and test sets
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [108]:
with mlflow.start_run():
    
   #Fit the pipeline on the training data
    pipeline.fit(X_train,y_train)

    #Predict on the test data
    preds=pipeline.predict(X_test)

    #Evaluate the model
    acc=accuracy_score(y_test,preds)
    mlflow.log_metric("accuracy", acc)
    print(f"Accuracy: {acc*100:.2f}%")
    
    #Log the model and metrics to MLflow
    #means that we are logging the accuracy metric with the name "accuracy" and the value of acc
    mlflow.log_metric("accuracy",acc)
    
    #means that we are logging a parameter with the name "model" and the value "GradientBoostingClassifier"
    mlflow.log_param("model",model_name)
    
    #Log the model to MLflow
    #means that we are logging the pipeline object as a model with the name "model"
    mlflow.sklearn.log_model(
       sk_model=pipeline,
       artifact_path="model",
       registered_model_name="bank-marketing-model")
    
    mlflow.set_tracking_uri("http://127.0.0.1:5000")
    mlflow.set_registry_uri("http://127.0.0.1:5000")
    mlflow.set_experiment("Default")

2026/04/03 19:10:40 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Accuracy: 91.89%


2026/04/03 19:10:40 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'bank-marketing-model' already exists. Creating a new version of this model...
2026/04/03 19:10:49 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: bank-marketing-model, version 2


🏃 View run dapper-seal-996 at: http://127.0.0.1:5000/#/experiments/0/runs/5c09b3d7a66449d095c863d8040f3b52
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/0


Created version '2' of model 'bank-marketing-model'.


In [109]:
df.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
